In [1]:
import pandas as pd
import numpy as np
import xml.etree.ElementTree as ET
import gzip
import unidecode
import gc
import os
from ipumspy import readers
import re
import ast

from tqdm import tqdm
tqdm.pandas()

In [2]:
ddi_path = "ipumsi_00002.xml"
data_path = "ipumsi_00002.dat.gz"

In [3]:
# ler xml como string
with open(ddi_path, "r", encoding="utf-8") as f:
    xml = f.read()

# remover bloco da variável CNTYAREAH
xml_clean = re.sub(
    r"<var[^>]*name=\"CNTYAREAH\"[\s\S]*?</var>",
    "",
    xml
)

# salvar xml limpo
clean_path = "ipums_clean.xml"
with open(clean_path, "w", encoding="utf-8") as f:
    f.write(xml_clean)

In [4]:
ddi = readers.read_ipums_ddi(clean_path)
ipums_df = readers.read_microdata(ddi, data_path)

/usr/local/lib/python3.11/site-packages/ipumspy/readers.py:70: CitationWarning: Use of data from IPUMS is subject to conditions including that users should cite the data appropriately.
See the `ipums_conditions` attribute of this codebook for terms of use.
See the `ipums_citation` attribute of this codebook for the appropriate citation.
  warnings.warn(


In [5]:
ipums_df.head()

,COUNTRY,YEAR,SAMPLE,SERIAL,PERSONS,HHWT,GQ,UNREL,URBAN,REGIONW,...,OC1HIRNO,OC1TRDNO,OC2HIRNO,OC2TRDNO,OCSTATUS,OCRELATE,SURSIM,NAMELAST,NAMEFRST,PTRESINO
0,578,1900,578190001,1000,3,1.0,10,0,9,42,...,27,32,99,99,99,0,1,Tønnes.,Peder Just,921
1,578,1900,578190001,1000,3,1.0,10,0,9,42,...,38,32,99,99,99,0,2,Jens. Tonnes.,Tonnes,921
2,578,1900,578190001,1000,3,1.0,10,0,9,42,...,27,32,99,99,99,0,3,Terjes.,Anton Gunerius,921
3,578,1900,578190001,2000,11,1.0,10,0,9,42,...,27,32,99,99,99,0,1,Hauge,O.,1213
4,578,1900,578190001,2000,11,1.0,10,0,9,42,...,27,32,99,99,99,0,2,Omunds.,Andreas S.,1106


In [6]:
ipums_df.columns

Index(['COUNTRY', 'YEAR', 'SAMPLE', 'SERIAL', 'PERSONS', 'HHWT', 'GQ', 'UNREL',
       'URBAN', 'REGIONW', 'ENUMDIST', 'GEO1_NO1801_1910', 'MUNICNO',
       'HHNBRNO', 'RESLSNO', 'HHTYPE', 'NFAMS', 'NCOUPLES', 'NMOTHERS',
       'NFATHERS', 'HEADLOC', 'PRMFAMSZ', 'PERNUM', 'PERWT', 'RESIDENT',
       'MOMLOC', 'POPLOC', 'SPLOC', 'SPRULEH', 'STEPMOMH', 'STEPPOPH',
       'MOMRULEH', 'POPRULEH', 'FAMUNIT', 'FAMSIZE', 'NCHILD', 'NCHLT5',
       'ELDCH', 'YNGCH', 'NSIBS', 'NUMGEN', 'NONRELS', 'RELATE', 'RELATED',
       'RELATEH', 'AGE', 'SEX', 'MARST', 'MARSTD', 'BIRTHYR', 'NATIVITY',
       'BPLCOUNTRY', 'NATION', 'BPLNO', 'LABFORCE', 'OCCISCO', 'ISCO68A',
       'OCCHISCO', 'PRODUCT', 'OC1HIRNO', 'OC1TRDNO', 'OC2HIRNO', 'OC2TRDNO',
       'OCSTATUS', 'OCRELATE', 'SURSIM', 'NAMELAST', 'NAMEFRST', 'PTRESINO'],
      dtype='object')

In [7]:
colunas_necessarias = [
    'SERIAL', 'PERNUM', 'GQ', 'RELATED', 'RELATE', 'NAMEFRST', 'NAMELAST', 'SAMPLE',
    'AGE', 'OCCISCO', 'ISCO68A', 'OCCHISCO', 'MUNICNO', 'BPLNO', 'GEO1_NO1801_1910'
]
ipums_df = ipums_df[colunas_necessarias]

# VALORES DE MISSING POR COLUNA
MISSING_VALUES = {
    'GQ': [99],
    'AGE': [999],
    'OCCISCO': [98, 99],
    'ISCO68A': [998, 999],
    'OCCHISCO': [99999],
    'RELATE': [9],
    'MUNICNO': [1699, 2099, 9997],
    'BPLNO': [1699, 2099, 9997, 9998, 9999],
    'GEO1_NO1801_1910': [578099]
}
for col, vals in MISSING_VALUES.items():
    if col in ipums_df.columns:
        ipums_df.loc[ipums_df[col].isin(vals), col] = np.nan


col_to_str = ['RELATE', 'RELATED', 'OCCISCO', 'ISCO68A', 'OCCHISCO','GEO1_NO1801_1910']
for col in col_to_str:
    var = next(v for v in ddi.data_description if v.name == col)
    var_map = {int(v): k for k, v in var.codes.items()}
    ipums_df[f"{col}_str"] = ipums_df[col].map(var_map)


df_muni = pd.read_csv("municipios_noruega.csv")
# Criar mapping {code -> label}
muni_map = dict(zip(df_muni['code'], df_muni['label']))
col_to_str = ['MUNICNO', 'BPLNO']
for col in col_to_str:
    ipums_df[f"{col}_str"] = ipums_df[col].map(muni_map)

ipums_df.shape

(2294599, 23)

In [8]:
cols_all_empty = ['AGE', 'OCCISCO', 'ISCO68A', 'OCCHISCO', 'MUNICNO', 'BPLNO']

mask_all_empty = (
    ipums_df[cols_all_empty].isna() |
    ipums_df[cols_all_empty].astype(str).apply(lambda x: x.str.strip() == "")
).all(axis=1)

ipums_df = ipums_df[~mask_all_empty]

ipums_df.shape

(2294590, 23)

In [9]:
# Criar ID único
ipums_df['person_id'] = ipums_df['SERIAL'].astype(str) + '_' + ipums_df['PERNUM'].astype(str)


ipums_df['nome_norm'] = (
    ipums_df['NAMEFRST'].fillna("").str.strip().replace(".", "") + " " +
    ipums_df['NAMELAST'].fillna("").str.strip().replace(".", "")
).str.upper().apply(unidecode.unidecode).str.strip()

In [10]:
# Agrupar por SERIAL e contar quantos Head
count_head = (
    ipums_df[ipums_df['RELATED_str'] == "Head"]
    .groupby('SERIAL')
    .size()
    .reset_index(name='count_head')
)

# Mostrar SERIALs com mais de 1 Head
count_head = count_head[count_head['count_head'] > 1]
serial_mais_um_head = count_head.SERIAL.unique()
print(len(serial_mais_um_head))

4630


In [11]:
df_chefes = ipums_df

# Remover residencias com mais de um Chefe
df_chefes = df_chefes[~df_chefes['SERIAL'].isin(serial_mais_um_head)]

# Apenas chefes
df_chefes = df_chefes[df_chefes['RELATED_str'] == "Head"]

# Filtrar GQ válidos
gq_desejados = [10, 20]
df_chefes = df_chefes[df_chefes['GQ'].isin(gq_desejados)]

# Chefes validos
valid_serials = df_chefes.SERIAL.unique()
len(valid_serials)

369713

In [12]:
df_parentes = ipums_df

# Remover residencias com mais de um Chefe
df_parentes = df_parentes[~df_parentes['SERIAL'].isin(serial_mais_um_head)]

# Apenas membros
df_parentes = df_parentes[df_parentes['RELATED_str'] != "Head"]

# Filtrar GQ válidos
gq_desejados = [10, 20]
df_parentes = df_parentes[df_parentes['GQ'].isin(gq_desejados)]

# Filtrar RELATED válidos
related_to_remove = [4600, 4800, 4900, 4910, 4920, 4930, 5000, 5100, 5110, 5111, 5112, 5113, 5120, 5130, 5140, 5150, 5200, 5210, 5220, 5221, 5222, 5223, 5300, 5310, 
5311, 5320, 5330, 5340, 5350, 5400, 5500, 5510,5520, 5600, 5610, 5620, 5900, 6000, 9999]
df_parentes = df_parentes[~df_parentes['RELATED'].isin(related_to_remove)]

# Apenas parentes com chefes validos
df_parentes = df_parentes[df_parentes['SERIAL'].isin(valid_serials)]

df_parentes.shape

(1319176, 25)

In [13]:
# Criar pares
df_pares = pd.merge(
    df_parentes,
    df_chefes,
    on='SERIAL',
    suffixes=('_parente', '_chefe')
)

cond_age = (
    df_pares['AGE_chefe'].notna() &
    df_pares['AGE_parente'].notna()
)

cond_loc = (
    (df_pares['MUNICNO_chefe'].notna() | df_pares['BPLNO_chefe'].notna()) &
    (df_pares['MUNICNO_parente'].notna() | df_pares['BPLNO_parente'].notna())
)

cond_occ = (
    (df_pares['OCCISCO_chefe'].notna() |
     df_pares['ISCO68A_chefe'].notna() |
     df_pares['OCCHISCO_chefe'].notna())
    &
    (df_pares['OCCISCO_parente'].notna() |
     df_pares['ISCO68A_parente'].notna() |
     df_pares['OCCHISCO_parente'].notna())
)

# Remove pares que o chefe o parente nao tem nenhum dado em comum
df_pares = df_pares[cond_age | cond_loc | cond_occ]

df_pares.shape

(1319176, 49)

In [14]:
# Buscar os homonimos dos parentes
homonimos = (
    ipums_df.groupby('nome_norm')['person_id']
    .apply(list)
    .to_dict()
)
df_pares['lista_homonimos'] = df_pares['nome_norm_parente'].map(homonimos)

In [15]:
# Remover pares com parente sem homonimo
df_pares = df_pares[df_pares['lista_homonimos'].str.len() > 1]

df_pares.shape

(783948, 50)

In [16]:
df_pares['lista_homonimos'].str.len().mean(), df_pares['lista_homonimos'].str.len().std()

(np.float64(178.73101532244485), np.float64(534.1276409629346))

In [17]:
df_pares.head()

,SERIAL,PERNUM_parente,GQ_parente,RELATED_parente,RELATE_parente,NAMEFRST_parente,NAMELAST_parente,SAMPLE_parente,AGE_parente,OCCISCO_parente,...,RELATED_str_chefe,OCCISCO_str_chefe,ISCO68A_str_chefe,OCCHISCO_str_chefe,GEO1_NO1801_1910_str_chefe,MUNICNO_str_chefe,BPLNO_str_chefe,person_id_chefe,nome_norm_chefe,lista_homonimos
0,2621000,2,10,2100,2,Fredrikke,Anderssen,578190001,62,<NA>,...,Head,Clerks,Government executive officials,"Civil servants, title unknown",Østfold,Halden,Oslo/Kristiana,2621000_1,RICHARD ANDERSSEN,"[2621000_2, 426449000_2]"
1,2622000,2,10,2100,2,Christiane,Dahl,578190001,56,<NA>,...,Head,Clerks,Government executive officials,Customs officers,Østfold,Halden,Berg,2622000_1,LARS DAHL,"[2622000_2, 56298000_1, 150660000_1, 382059000_4]"
3,2622000,4,10,3100,3,Johannes,Dahl,578190001,17,7,...,Head,Clerks,Government executive officials,Customs officers,Østfold,Halden,Berg,2622000_1,LARS DAHL,"[1051000_13, 2622000_4, 2814000_8, 7779000_4, ..."
4,2624000,2,10,2100,2,Thrine,Berg,578190001,68,<NA>,...,Head,Technicians and associate professionals,Ships' deck officers and pilots,Ship's masters and captains (sea or inland wat...,Østfold,Halden,Halden,2624000_1,MATHIAS BERG,"[2624000_2, 365476000_2]"
5,2624000,3,10,3100,3,Petra,Berg,578190001,28,4,...,Head,Technicians and associate professionals,Ships' deck officers and pilots,Ship's masters and captains (sea or inland wat...,Østfold,Halden,Halden,2624000_1,MATHIAS BERG,"[2624000_3, 20365000_3, 66423000_5, 109713000_..."


In [18]:
pessoas = ipums_df.set_index("person_id", drop=False).to_dict("index")

In [19]:
def filtrar_lista(row):
    list_result = []
    for r_id in row["lista_homonimos"]:
        r = pessoas[r_id]
        e = r_id
        if not (pd.notna(row["AGE_chefe"]) and pd.notna(r["AGE"])):
            if not (
                (pd.notna(row["MUNICNO_chefe"]) or pd.notna(row["BPLNO_chefe"])) and
                (pd.notna(r["MUNICNO"]) or pd.notna(r["BPLNO"]))
            ):
                if not (
                    (pd.notna(row["OCCISCO_chefe"]) or pd.notna(row["ISCO68A_chefe"]) or pd.notna(row["OCCHISCO_chefe"])) and
                    (pd.notna(r["OCCISCO"]) or pd.notna(r["ISCO68A"]) or pd.notna(r["OCCHISCO"]))
                ):
                    e = None
        if e is not None:
            list_result.append(e)

    return list_result


df_pares["lista_homonimos"] = df_pares.progress_apply(filtrar_lista, axis=1)

100%|██████████| 783948/783948 [10:04<00:00, 1296.06it/s]


In [20]:
# Remover pares com parente sem homonimo apos validacao dos homonimos
df_pares = df_pares[df_pares['lista_homonimos'].str.len() > 1]

df_pares.shape

(783948, 50)

In [21]:
df_pares['lista_homonimos'].str.len().mean(), df_pares['lista_homonimos'].str.len().std()

(np.float64(178.73097960579017), np.float64(534.1275962763038))

In [22]:
row = df_pares.iloc[5]
print(row.nome_norm_chefe, row.nome_norm_parente, row.RELATED_str_parente, row.person_id_parente)

for p in row.lista_homonimos:
    r = pessoas[p]
    print(r['nome_norm'], p)

row

JENS THEODOR HALVORSEN OLAVA HALVORSEN Spouse 2627000_2
OLAVA HALVORSEN 2627000_2
OLAVA HALVORSEN 20193000_2
OLAVA HALVORSEN 20367000_2
OLAVA HALVORSEN 28926000_2
OLAVA HALVORSEN 34705000_1
OLAVA HALVORSEN 34744000_6
OLAVA HALVORSEN 38754000_1
OLAVA HALVORSEN 47345000_2
OLAVA HALVORSEN 62271000_2
OLAVA HALVORSEN 76466000_2
OLAVA HALVORSEN 77198000_8
OLAVA HALVORSEN 96195000_2
OLAVA HALVORSEN 104121000_9
OLAVA HALVORSEN 107955000_2
OLAVA HALVORSEN 168823000_2
OLAVA HALVORSEN 184606000_5
OLAVA HALVORSEN 200190000_1
OLAVA HALVORSEN 200327000_2


SERIAL                                                                    2627000
PERNUM_parente                                                                  2
GQ_parente                                                                     10
RELATED_parente                                                              2100
RELATE_parente                                                                  2
NAMEFRST_parente                                                            Olava
NAMELAST_parente                                                        Halvorsen
SAMPLE_parente                                                          578190001
AGE_parente                                                                    53
OCCISCO_parente                                                              <NA>
ISCO68A_parente                                                              <NA>
OCCHISCO_parente                                                             <NA>
MUNICNO_parente 

In [23]:
df_pares["RELATED"] = df_pares["RELATED_str_parente"]
columns = ["person_id_chefe", "person_id_parente", "RELATED", "lista_homonimos"]
df_pares = df_pares[columns]

In [24]:
df_pares.RELATED.value_counts()

RELATED
Biological child     543817
Spouse               203458
Adopted child          9451
Parent                 6639
Grandchild             6617
Parent-in-law          5097
Sibling                3852
Child-in-law           2036
Grandparent             998
Stepchild               770
Nephew/niece            725
Sibling-in-law          304
Stepparent              118
Cousin                   26
Stepsibling              25
Great grandchild         13
Unmarried partner         2
Name: count, dtype: int64

In [25]:
# remover tipos de relacionamento com menos de 100 pares
df_pares=df_pares[~df_pares.RELATED.isin(["Cousin", "Stepsibling", "Great grandchild", "Unmarried partner"])]

In [26]:
df_pares = df_pares.sample(frac=1, random_state=42).reset_index(drop=True)
df_pares["ID"] = range(1, len(df_pares) + 1)

In [27]:
df_pares['FOLD'] = -1

# Dividir índices em 5 partes aproximadamente iguais
folds = np.array_split(df_pares.index, 5)

# Atribuir número do fold
for i, idx in enumerate(folds):
    df_pares.loc[idx, 'FOLD'] = i + 1

# Conferir quantidade por fold
print(df_pares['FOLD'].value_counts())

print("TOtal", len(df_pares))

FOLD
1    156777
2    156777
3    156776
4    156776
5    156776
Name: count, dtype: int64
TOtal 783882


In [28]:
df_pares.to_pickle("IpumsNORWAY_pares.pkl")

In [29]:
del df_pares
del df_parentes
del df_chefes

gc.collect()

55

In [30]:
df_pares = pd.read_pickle("IpumsNORWAY_pares.pkl")

In [31]:
person_id_chefe_unique = np.unique(df_pares["person_id_chefe"].dropna().values)

In [32]:
person_id_parente_unique = np.unique(df_pares["person_id_parente"].dropna().values)

In [33]:
homonimos_id_unique = set()

for lst in df_pares["lista_homonimos"]:
    if isinstance(lst, list):
        homonimos_id_unique.update(lst)

homonimos_id_unique = np.array(list(homonimos_id_unique))

In [34]:
unique_ids = np.unique(
    np.concatenate([
        person_id_chefe_unique,
        person_id_parente_unique,
        homonimos_id_unique
    ])
)

In [35]:
ipums_df = ipums_df[ipums_df["person_id"].isin(unique_ids)]

In [36]:
ipums_df['OCC_ID'] = (
    ipums_df[['OCCISCO','ISCO68A','OCCHISCO']]
    .astype('string')
    .fillna('')
    .agg('_'.join, axis=1)
)

ipums_df['OCC_str'] = (
    ipums_df[['OCCISCO_str','ISCO68A_str','OCCHISCO_str']]
    .astype('string')
    .fillna('')
    .agg('. '.join, axis=1)
)

ipums_df = ipums_df[["person_id","nome_norm","AGE","MUNICNO","BPLNO","MUNICNO_str","BPLNO_str", "OCC_ID", "OCC_str"]]

In [37]:
ipums_df.to_csv("IpumsNORWAY_distinct_person.csv", index=False)

In [38]:
df_occ = ipums_df[["OCC_ID", "OCC_str"]].drop_duplicates()
df_occ = df_occ[df_occ["OCC_ID"] != "__"]
df_occ["template"] = "Occupation: " + df_occ["OCC_str"] 
df_occ.to_csv("IpumsNORWAY_distinct_profissao.csv", index=False)

In [39]:
df_addr = ipums_df[["MUNICNO", "MUNICNO_str"]].drop_duplicates()
df_addr["MUNI_ID"] = df_addr["MUNICNO"]
df_addr["MUNI_str"] = df_addr["MUNICNO_str"]

In [40]:
df_addr_2 = ipums_df[["BPLNO", "BPLNO_str"]].drop_duplicates()
df_addr_2["MUNI_ID"] = df_addr_2["BPLNO"]
df_addr_2["MUNI_str"] = df_addr_2["BPLNO_str"]

In [41]:
df_addr_concat = pd.concat(
    (df_addr[["MUNI_ID", "MUNI_str"]],
     df_addr_2[["MUNI_ID", "MUNI_str"]]),
    ignore_index=True
).drop_duplicates(ignore_index=True)

In [42]:
df_addr_concat["template"] = df_addr_concat["MUNI_str"] + " kommune, Norge"
df_addr_concat.to_csv("IpumsNORWAY_distinct_municipio.csv", index=False)